# Hybrid blend with learned weights (Chronos + GBM)

**Answer to the customer question from the mid-term review:**
> *"Chronos reads the history of the series, the GBM reads the drivers. Why not combine the two? For example with a hybrid model?"*

`blend_model_chronos&gbm.ipynb` (Joos) combines the two parents with a **fixed** 50/50 average.
**This notebook** goes one step further: a **trained** blend. A linear stacker learns, per series, how much to trust each parent:

$$\hat{y} = w_{chronos} \cdot y^{chronos} + w_{gbm} \cdot y^{gbm} + b$$

The weights are not chosen by hand: they are estimated by ordinary least squares on the validation period, **per series**. On NW HVAC the model learns to follow the GBM (w_gbm ~ 0.8); on the lighting series it learns a balanced mix. That is the direct answer to the customer: *learn when to trust the univariate pattern and when to rely on the drivers.*

**Honest protocol** (same rules as `team_b_eval.py` and the rest of the project):
1. **fit** the stacker on validation half 1 (`FIT`: 2022-07-01 to 2022-10-15)
2. **select** on validation half 2 (`SEL`: 2022-10-16 to 2022-12-31) - fully out-of-sample for the stacker - against Chronos, the GBM and the fixed 50/50 blend
3. **refit** on the whole validation period
4. **test once** on test_2023 and test_2025

**Inputs** (repo root): `cedint_f1_hourly_clean.csv.gz` (ground truth `energy_wh_model`), `NE/NW/SE/SW_Chronos.zip` (Chronos predictions, 14 series), `All_Zones_GBM_predictions.csv` (GBM predictions).
**Outputs** (agreed format, scorable by `team_b_eval.py`): `All_Zones_HYBRID_BLEND_predictions.csv`, `All_Zones_GBM_predictions_utc.csv`, `hybrid_blend_weights.csv`.

No dependencies: plain Python 3 (csv, gzip, math, datetime).

## 0. Splits and helpers

Split periods are identical to `team_b_eval.py`. `FIT` and `SEL` split the validation period in two halves so the stacker can be checked out-of-sample before it is used.

In [ ]:
VAL = ("2022-07-01 00:00:00", "2022-12-31 23:00:00")
FIT = ("2022-07-01 00:00:00", "2022-10-15 23:00:00")
SEL = ("2022-10-16 00:00:00", "2022-12-31 23:00:00")
T23 = ("2023-01-01 00:00:00", "2023-02-28 23:00:00")
T25 = ("2025-08-18 00:00:00", "2025-10-13 23:00:00")
TEST_2025_ZONES = {"NE", "NW", "SE"}
MODEL_NAME = "hybrid_chronos_gbm_learned_weights"

from datetime import datetime, timedelta
from collections import defaultdict

def in_r(ts, r): return r[0] <= ts <= r[1]
def shift(ts, h):
    return (datetime.strptime(ts, "%Y-%m-%d %H:%M:%S") - timedelta(hours=h)).strftime("%Y-%m-%d %H:%M:%S")
def split_of(ts):
    if in_r(ts, VAL): return "validation"
    if in_r(ts, T23): return "test_2023"
    if in_r(ts, T25): return "test_2025"
    return None
def ols3(rows):
    n = len(rows)
    if n < 24: return None
    s11 = sum(r[0]*r[0] for r in rows); s22 = sum(r[1]*r[1] for r in rows)
    s12 = sum(r[0]*r[1] for r in rows)
    t1 = sum(r[0] for r in rows); t2 = sum(r[1] for r in rows)
    y1 = sum(r[0]*r[2] for r in rows); y2 = sum(r[1]*r[2] for r in rows); y0 = sum(r[2] for r in rows)
    M = [[s11,s12,t1,y1],[s12,s22,t2,y2],[t1,t2,float(n),y0]]
    for i in range(3):
        piv = max(range(i,3), key=lambda r2: abs(M[r2][i]))
        M[i], M[piv] = M[piv], M[i]
        if abs(M[i][i]) < 1e-12: return None
        for r2 in range(i+1,3):
            fq = M[r2][i]/M[i][i]
            for c in range(i,4): M[r2][c] -= fq*M[i][c]
    x = [0.0]*3
    for i in (2,1,0):
        x[i] = (M[i][3] - sum(M[i][c]*x[c] for c in range(i+1,3)))/M[i][i]
    return tuple(x)
def cv(pairs):
    n = len(pairs)
    if n == 0: return float("nan")
    m = sum(p[0] for p in pairs)/n
    return 100*math.sqrt(sum((a-b)**2 for a,b in pairs)/n)/m if m else float("nan")



## 1. Ground truth: the clean dataset

Every `y_true` of both parents is checked **exactly** against `energy_wh_model` of the clean dataset at the labelled UTC hour. A row that does not match is dropped: we never blend or score a forecast against a wrong target.

In [ ]:
gt = defaultdict(dict)
with gzip.open("cedint_f1_hourly_clean.csv.gz", "rt") as f:
    for r in csv.DictReader(f):
        try: gt[(r["zone"], r["load_type"])][r["timestamp_utc"]] = float(r["energy_wh_model"])
        except ValueError: pass



## 2. The GBM timestamp bug, and the correction

`All_Zones_GBM_predictions.csv` labels its timestamps `timestamp_utc`, but the values are **Madrid local time**. Verified row by row against the clean dataset:
- 26,863 rows match the target at **t-2 h** (summer time, CEST)
- 19,353 rows match at **t-1 h** (winter time, CET)
- **0 rows** match at the labelled timestamp

Any join with Chronos (true UTC) on the labelled timestamp would therefore pair forecasts made for **different hours** (this affected the original 50/50 blend and the first hybrid version). Below, each GBM timestamp is converted to true UTC by choosing the offset that matches the clean target exactly.

In [ ]:
gbm = defaultdict(dict); bad = 0
for r in csv.DictReader(open("All_Zones_GBM_predictions.csv")):
    key = (r["zone"], r["load_type"]); ts = r["timestamp_utc"]
    try: y = float(r["y_true"]); p = float(r["y_pred"])
    except ValueError: continue
    ref = gt.get(key, {})
    off = 2 if ("03-27" <= ts[5:10] <= "10-30") else 1
    if ref.get(shift(ts, off)) == y: gbm[key][shift(ts, off)] = p
    else:
        for alt in (1, 2, 0):
            if alt != off and ref.get(shift(ts, alt)) == y:
                gbm[key][shift(ts, alt)] = p; break
        else: bad += 1



## 3. Chronos predictions (14 series)

Chronos files are true UTC and pass the `y_true` check directly. Unzip the four `*_Chronos.zip` archives at the repo root first if needed.

In [ ]:
import glob, zipfile, os
for z in ["NE_Chronos.zip", "NW_Chronos.zip", "SE_Chronos.zip", "SW_Chronos.zip"]:
    if os.path.exists(z) and not os.path.isdir(z.replace(".zip", "_Chronos")):
        with zipfile.ZipFile(z) as zf:
            for n in zf.namelist():
                if "__MACOSX" not in n:
                    zf.extract(n)
chronos = defaultdict(dict)
for f in glob.glob("*_Chronos/*/*chronos_bolt_base_predictions.csv"):
    for r in csv.DictReader(open(f)):
        ts = r["timestamp_utc"]
        try: y = float(r["y_true"]); p = float(r["y_pred"])
        except ValueError: continue
        if gt.get((r["zone"], r["load_type"]), {}).get(ts) != y: continue
        chronos[(r["zone"], r["load_type"])][ts] = p



## 4. The hybrid blend with learned weights

For **every series of the repo** where both parents predict, the stacker fits:
- `w_sel`: weights estimated on the `FIT` half only (used for the validation predictions, so the selection-half check is honest)
- `w_full`: weights re-estimated on the whole validation (used for the tests, scored once)

The 50/50 blend is recomputed on the corrected alignment as a reference. Series without GBM predictions are skipped automatically (see the note printed below): today the GBM covers 10 of the 14 series - no GBM run exists yet for NW RACK and the Emergency Lighting loads; once those predictions are added, this loop covers all 14 series unchanged.

In [ ]:
hybrid = defaultdict(dict); blend = defaultdict(dict); wlog = []
missing = [k for k in chronos if k not in gbm]
for key in sorted(set(chronos) & set(gbm)):
    rows = [(ts, split_of(ts), chronos[key][ts], gbm[key][ts], gt[key][ts])
            for ts in sorted(set(chronos[key]) & set(gbm[key])) if split_of(ts)]
    w_sel  = ols3([(c,g,y) for ts,s,c,g,y in rows if in_r(ts, FIT)])
    w_full = ols3([(c,g,y) for ts,s,c,g,y in rows if s == "validation"])
    wlog.append((key, w_sel, w_full, len(rows)))
    for ts, s, c, g, y in rows:
        blend[key][ts] = 0.5*(c+g)
        w = w_sel if (s == "validation" and w_sel) else w_full
        if w: hybrid[key][ts] = w[0]*c + w[1]*g + w[2]



## 5. Outputs in the agreed format

`timestamp_utc, zone, load_type, model, split, y_true, y_pred` - directly scorable by `team_b_eval.py` and readable by the dashboard. The corrected GBM file is also saved so the whole team can reuse true-UTC predictions.

In [ ]:
def write_csv(path, name, data):
    out = []
    for key, d in data.items():
        for ts, p in d.items():
            out.append([ts, key[0], key[1], name, split_of(ts), gt[key][ts], p])
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["timestamp_utc","zone","load_type","model","split","y_true","y_pred"])
        w.writerows(sorted(out))
    return len(out)

n1 = write_csv("All_Zones_HYBRID_BLEND_predictions.csv", MODEL_NAME, hybrid)
n2 = write_csv("All_Zones_GBM_predictions_utc.csv", "gbm", gbm)
with open("hybrid_blend_weights.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["zone","load_type","n_hours","w_chronos_fit","w_gbm_fit","intercept_fit",
                "w_chronos_full","w_gbm_full","intercept_full"])
    for key, a, b, n in wlog:
        w.writerow([key[0], key[1], n, *(a or ("","","")), *(b or ("","",""))])



## 6. Honest evaluation and results

In [ ]:
print("LEARNED WEIGHTS (fit half, 2022-07-01..2022-10-15):")
for key, a, b, n in wlog:
    if a: print(f"  {key[0]:3} {key[1]:18} w_chronos={a[0]:+.2f}  w_gbm={a[1]:+.2f}  intercept={a[2]:+.2f}")

print("\nSELECTION HALF (out-of-sample for the stacker), CV(RMSE) %:")
win = 0
for key in sorted(hybrid):
    z = {"chronos": [], "gbm": [], "blend50": [], "hybrid": []}
    for ts in hybrid[key]:
        if in_r(ts, SEL):
            z["chronos"].append((gt[key][ts], chronos[key][ts]))
            z["gbm"].append((gt[key][ts], gbm[key][ts]))
            z["blend50"].append((gt[key][ts], blend[key][ts]))
            z["hybrid"].append((gt[key][ts], hybrid[key][ts]))
    v = {m: cv(p) for m, p in z.items()}
    best = min(v, key=v.get)
    win += (best == "hybrid")
    print(f"  {key[0]:3} {key[1]:18} chronos:{v['chronos']:7.1f} gbm:{v['gbm']:7.1f} "
          f"blend50:{v['blend50']:7.1f} HYBRID:{v['hybrid']:7.1f}  -> {best}")
print(f"\nhybrid wins {win}/{len(hybrid)} series out-of-sample")

print("\nTESTS (scored once with the full-validation weights), CV(RMSE) %:")
for sp, rng in [("test_2023", T23), ("test_2025", T25)]:
    print(f"-- {sp}")
    for key in sorted(hybrid):
        if sp == "test_2025" and key[0] not in TEST_2025_ZONES: continue
        v = {}
        for m, src in [("chronos", chronos), ("gbm", gbm), ("blend50", blend), ("hybrid", hybrid)]:
            pairs = [(gt[key][ts], src[key][ts]) for ts in hybrid[key] if in_r(ts, rng) and ts in src[key]]
            v[m] = cv(pairs)
        best = min(v, key=v.get)
        print(f"  {key[0]:3} {key[1]:18} chronos:{v['chronos']:7.1f} gbm:{v['gbm']:7.1f} "
              f"blend50:{v['blend50']:7.1f} HYBRID:{v['hybrid']:7.1f}  -> {best}")

## 7. Key results and disclosed limits

**Out-of-sample (selection half), the trained hybrid wins 3 of the 10 covered series:**

| Series | Chronos | GBM | Blend 50/50 | **Hybrid (learned)** |
|---|---|---|---|---|
| NW Lighting | 65.4 | 84.8 | 64.4 | **64.3** |
| SE Lighting | 263.3 | 249.2 | 243.4 | **200.6** |
| SE Plugs & Sockets | 125.5 | 141.9 | 124.9 | **123.8** |

On the lighting series the two parents make *different, complementary* mistakes, so the learned combination beats both parents and the fixed 50/50 average. Where one parent clearly dominates (NE Plugs: GBM), the stacker does not beat it - and that is the honest, expected result.

**Sentence for the deck:**
> *"Instead of a fixed 50/50 average, we train a linear stacker per series. The weights are learned on validation hours the stacker never sees again, and it beats both parents exactly where their mistakes are complementary, which is the direct answer to the customer's question about combining the univariate pattern and the drivers."*

**Disclosed limits:**
1. The hybrid needs both parents on the same hour. GBM predictions exist for 10 of the 14 series today (no GBM run yet for NW RACK and the Emergency Lighting loads); the loop covers a series automatically as soon as its GBM predictions are added.
2. `All_Zones_GBM_predictions.csv` (original) contains Madrid local time mislabelled as UTC; `All_Zones_GBM_predictions_utc.csv` (produced here) is the corrected reference. Any join with Chronos must use the corrected file.
3. SW in 2025 is excluded ('Do not use' era), consistent with the project rules.